### Data Ingestion

###document data structure

In [64]:
from langchain_core.documents import Document

In [65]:
doc = Document(
    page_content="This is the main text content I am using to create RAG",
    metadata={
            "source": "example.txt",
            "pages": 1,
            "author": "Shivam Kumar",
            "date_created": "2026-03-22"
    },
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Shivam Kumar', 'date_created': '2026-03-22'}, page_content='This is the main text content I am using to create RAG')

In [66]:
## Create a simple text file

import os
os.makedirs("../data/text_files",exist_ok=True)

In [67]:
sample_texts={
    "../data/text_files/python_intro.txt": """Python Programming Introduction

    Python is a high-level, interpreted programming language known for its simplicity and readability. It was created by Guido van Rossum and first released in 1991. Python's design philosophy emphasizes code readability, making it an excellent choice for beginners and experienced developers alike.""",
    "../data/text_files/machine_learning.txt": """Machine Learning Basics
    
    Machine learning is a subset of artificial intelligence that focuses on enabling computers to learn from data and make predictions or decisions without being explicitly programmed. It involves algorithms that can identify patterns in data and improve their performance over time."""
}

for file_path, content in sample_texts.items():
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)

print("Sample text files created successfully.")

Sample text files created successfully.


In [68]:
### TextLoader

from langchain_community.document_loaders import TextLoader

loader= TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document = loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content="Python Programming Introduction\n\n    Python is a high-level, interpreted programming language known for its simplicity and readability. It was created by Guido van Rossum and first released in 1991. Python's design philosophy emphasizes code readability, making it an excellent choice for beginners and experienced developers alike.")]


In [69]:
### DirectoryLoader

from langchain_community.document_loaders import DirectoryLoader
dir_loader = DirectoryLoader(
        "../data/text_files",
        glob= "**/*.txt",
        loader_cls = TextLoader,
        loader_kwargs={'encoding': 'utf-8'},
        show_progress=False
)

documents=dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\n    Machine learning is a subset of artificial intelligence that focuses on enabling computers to learn from data and make predictions or decisions without being explicitly programmed. It involves algorithms that can identify patterns in data and improve their performance over time.'),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content="Python Programming Introduction\n\n    Python is a high-level, interpreted programming language known for its simplicity and readability. It was created by Guido van Rossum and first released in 1991. Python's design philosophy emphasizes code readability, making it an excellent choice for beginners and experienced developers alike.")]

In [70]:
### PDFLoader

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
dir_loader = DirectoryLoader(
        "../data/pdf",
        glob= "**/*.pdf",
        loader_cls = PyMuPDFLoader,
        show_progress=False
)

pdf_documents=dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-03-11T07:53:21+00:00', 'source': '..\\data\\pdf\\Shivam_resume_2025.pdf', 'file_path': '..\\data\\pdf\\Shivam_resume_2025.pdf', 'total_pages': 1, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-03-11T07:53:21+00:00', 'trapped': '', 'modDate': 'D:20250311075321Z', 'creationDate': 'D:20250311075321Z', 'page': 0}, page_content='Shivam Kumar\nÓ +91 7017903175\nR 1306shivam@gmail.com\n° kumar-shivam2028\n\x89 ShivamKumar-mnnit\n\x0b Portfolio\nAcademic Qualifications\nMotilal Nehru National Institute of Technology Allahabad, Prayagraj\nUttar Pradesh, India\nBachelor Of Technology in Electrical Engineering | CPI: 8.66/10.0\n2021-2025\nSaraswati Vidya Mandir Senior Secondary School, Etah\nUttar Pradesh, India\nSenior Secondary, Central Board Of Secondary Education | Percentage: 96.6%\n2020\nSecondary, Central Board Of Secondary Education | P

### embeddings And VectorStoreDB

In [71]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity



In [72]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer."""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the EmbeddingManager with a SentenceTransformer model.
        Args:
            model_name : Hugging Face model name for sentence embeddings.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Loaded embedding model: {self.model_name}. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.
        Args:
            texts : List of text strings to embed.
        Returns:
            np.ndarray: Array of embedding vectors.
            numpy array of embeddings with shape(len(texts), embedding_dimension)
        """
        if self.model is None:
            raise ValueError("Embedding model is not loaded.")
        try:
            print(f"Generating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"Generated embeddings with shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise


    def get_embedding_dimension(self) -> int:
        """Get the dimension of the embeddings produced by the model."""
        if self.model is None:
            raise ValueError("Embedding model is not loaded.")
        return self.model.get_sentence_embedding_dimension()
    

## initailize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1281.81it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded embedding model: all-MiniLM-L6-v2. Embedding dimension: 384


### VectorStore

In [73]:
class VectorStore:
    """Manages document embeddings in a chormadb vector store."""
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the VectorStore.
        Args:
            collection_name : Name of the ChromaDB collection to use.
            persist_directory : Directory to persist the ChromaDB data.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the ChromaDB client and collection."""
        try:
            print("Initializing ChromaDB client...")
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create the collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
                )
            print(f"Vector store initialized with collection: {self.collection_name}")
            print(f"Existing Documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Document], embeddings: np.ndarray):
        """
        Add documents to the vector store after generating embeddings.
        Args:
            documents : List of Document objects to add.
            embeddings : Corresponding embeddings for the documents.
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        print(f"Adding {len(documents)} documents to the vector store...")

        # Prepare data for insertion
        ids=[]
        metadatas=[]
        document_texts=[]
        embeddings_list=[]

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #generate unique id for each document
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prepare metadata for the document
            metadata=dict(doc.metadata) 
            metadata['doc_index']=i
            metadata['context_length']=len(doc.page_content)
            metadatas.append(metadata)

            #Document Content
            document_texts.append(doc.page_content)

            #Embeddings
            embeddings_list.append(embedding.tolist())

        #Add to collection
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=document_texts,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection after addition: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore

Initializing ChromaDB client...
Vector store initialized with collection: pdf_documents
Existing Documents in collection: 12


In [74]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # Adjust based on your needs
    chunk_overlap=200,  # Overlap between chunks
    separators=["\n\n", "\n", " ", ""]  # Split on paragraphs, lines, words
)

# Split the documents into chunks
chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
chunks

Created 2 chunks from 2 documents


[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\n    Machine learning is a subset of artificial intelligence that focuses on enabling computers to learn from data and make predictions or decisions without being explicitly programmed. It involves algorithms that can identify patterns in data and improve their performance over time.'),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content="Python Programming Introduction\n\n    Python is a high-level, interpreted programming language known for its simplicity and readability. It was created by Guido van Rossum and first released in 1991. Python's design philosophy emphasizes code readability, making it an excellent choice for beginners and experienced developers alike.")]

In [75]:
# Chunk the PDF documents
pdf_chunks = text_splitter.split_documents(pdf_documents)

print(f"Created {len(pdf_chunks)} chunks from {len(pdf_documents)} PDF documents")
pdf_chunks

Created 6 chunks from 1 PDF documents


[Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-03-11T07:53:21+00:00', 'source': '..\\data\\pdf\\Shivam_resume_2025.pdf', 'file_path': '..\\data\\pdf\\Shivam_resume_2025.pdf', 'total_pages': 1, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-03-11T07:53:21+00:00', 'trapped': '', 'modDate': 'D:20250311075321Z', 'creationDate': 'D:20250311075321Z', 'page': 0}, page_content='Shivam Kumar\nÓ +91 7017903175\nR 1306shivam@gmail.com\n° kumar-shivam2028\n\x89 ShivamKumar-mnnit\n\x0b Portfolio\nAcademic Qualifications\nMotilal Nehru National Institute of Technology Allahabad, Prayagraj\nUttar Pradesh, India\nBachelor Of Technology in Electrical Engineering | CPI: 8.66/10.0\n2021-2025\nSaraswati Vidya Mandir Senior Secondary School, Etah\nUttar Pradesh, India\nSenior Secondary, Central Board Of Secondary Education | Percentage: 96.6%\n2020\nSecondary, Central Board Of Secondary Education | P

In [76]:
###convert text to embeddings
texts =[doc.page_content for doc in pdf_chunks]
texts

['Shivam Kumar\nÓ +91 7017903175\nR 1306shivam@gmail.com\n° kumar-shivam2028\n\x89 ShivamKumar-mnnit\n\x0b Portfolio\nAcademic Qualifications\nMotilal Nehru National Institute of Technology Allahabad, Prayagraj\nUttar Pradesh, India\nBachelor Of Technology in Electrical Engineering | CPI: 8.66/10.0\n2021-2025\nSaraswati Vidya Mandir Senior Secondary School, Etah\nUttar Pradesh, India\nSenior Secondary, Central Board Of Secondary Education | Percentage: 96.6%\n2020\nSecondary, Central Board Of Secondary Education | Percentage: 94.4%\n2018\nExperience\nSoftware Engineering Intern | Deutsche Telekom Digital Labs, Gurugram\nJan 2025 - present\n• Working in one-shop team, increase the tested code coverage upto 90% (junit and SonarQube)\nFull Stack Development Project Mentorship Intern | BLACK BUCKS,Jubilee Hills,\nJan 2024 - Apr 2024\nHyd,TG500033IN\n• Worked with the tech team to extract project abstracts from GitHub, delivering over 10 abstracts per day.\nWeb Developer | Robotics Club MNN

In [77]:
## Generate embeddings for the chunks
embeddings = embedding_manager.generate_embeddings(texts)

##store in vector database
vectorstore.add_documents(pdf_chunks, embeddings)

Generating embeddings for 6 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.15it/s]

Generated embeddings with shape: (6, 384)
Adding 6 documents to the vector store...
Successfully added 6 documents to the vector store.
Total documents in collection after addition: 18


### Retriever Pipeline From VectorStore

In [85]:
class RagRetriever:
    """Retrieves relevant document chunks from the vector store based on a query."""
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the RagRetriever.
        Args:
            vector_store : Instance of the VectorStore to retrieve from.
            embedding_manager : Instance of the EmbeddingManager to generate query embeddings.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5,score_threshold: float = 0.0) -> List[Dict]:
        """
        Retrieve relevant document chunks based on the query.
        Args:
            query : The input query string.
            top_k : Number of top relevant documents to retrieve.
            score_threshold : Minimum cosine similarity score to consider a document relevant.
        Returns:
            List of dictionaries containing the retrieved documents and metadata.
        """

        print(f"Retrieving relevant documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}...")
        
        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # process results
            retrieved_docs=[]

            if results['documents'] and results['documents'][0]:
                documents= results["documents"][0]
                metadatas= results["metadatas"][0]
                distances= results["distances"][0]
                ids= results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    #convert distance to similarity score (chormadb uses cosine distance)
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i+1
                        })
                print(f"Retrieved {len(retrieved_docs)} relevant documents for the query.")
            else:
                print("No relevant documents found for the query.")
            
            return retrieved_docs

        except Exception as e:
            print(f"Error retrieving documents from vector store: {e}")
            return []


rag_retriever = RagRetriever(vectorstore,embedding_manager)
rag_retriever

        

In [87]:
rag_retriever.retrieve("Academic Material Exchange Portal")

Retrieving relevant documents for query: 'Academic Material Exchange Portal' with top_k=5 and score_threshold=0.0...
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 26.71it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 relevant documents for the query.


[]

In [88]:
# === PDF query debug helper ===
print('PDF chunk count:', len(pdf_chunks) if 'pdf_chunks' in globals() else 'not defined')
print('Collection count:', vectorstore.collection.count())

query_text = 'Academic Material Exchange Portal'
query_embedding = embedding_manager.generate_embeddings([query_text])[0]

raw = vectorstore.collection.query(
    query_embeddings=[query_embedding],
    n_results=10,
    include=['documents', 'embeddings', 'metadatas', 'distances']
)

print('Raw results count:', len(raw.get('documents',[[]])[0]))
for i, doc in enumerate(raw.get('documents',[[]])[0]):
    dist = raw['distances'][0][i]
    print(f"[{i}] id={raw['ids'][0][i]} dist={dist:.4f} sim={1-dist:.4f}")
    print('doc preview:', doc[:250].replace('\n',' '))

# Drain threshold issue from retriever class
retriever_results = rag_retriever.retrieve(query_text, top_k=10, score_threshold=0.0)
print('RagRetriever results (threshold=0.0):', len(retriever_results))
for r in retriever_results:
    print('- rank', r['rank'], 'score', r['similarity_score'])
    print(' summary:', r['content'][:200].replace('\n',' '))

PDF chunk count: 6
Collection count: 18
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 56.72it/s]


Generated embeddings with shape: (1, 384)
Raw results count: 10
[0] id=doc_bbfe048c_3 dist=1.3660 sim=-0.3660
doc preview: advanced features like timed quizzes, result management and incorporating additional functionalities such as Google Authentication, password reset with OTP, and proctoring for a seamless online quiz experience. • Utilized technologies such as MERN St
[1] id=doc_7dc6ee7d_3 dist=1.3660 sim=-0.3660
doc preview: advanced features like timed quizzes, result management and incorporating additional functionalities such as Google Authentication, password reset with OTP, and proctoring for a seamless online quiz experience. • Utilized technologies such as MERN St
[2] id=doc_401b6466_3 dist=1.3660 sim=-0.3660
doc preview: advanced features like timed quizzes, result management and incorporating additional functionalities such as Google Authentication, password reset with OTP, and proctoring for a seamless online quiz experience. • Utilized technologies such as MERN St
[3] id

Batches: 100%|██████████| 1/1 [00:00<00:00, 65.71it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 relevant documents for the query.
RagRetriever results (threshold=0.0): 0
